# Docflow — dots.mocr inference server (Colab dev tool)

Runs `dots.ocr` on Colab's GPU and exposes a `/parse` endpoint that the local Docflow
engine calls via `RemoteDotsModel`. This is a **throwaway development tool** for Phase 0 —
production inference becomes a Modal function in Phase 1.

**Usage**
1. Runtime → Change runtime type → **GPU** (T4 is enough).
2. Run every cell top to bottom.
3. The last cell prints a public URL. On your machine set it and run the CLI:
   - PowerShell: `$env:DOCFLOW_DOTS_URL="<url>"`
   - then: `uv run docflow convert file.pdf --model dots -o out/`

**Debugging:** if `/parse` returns 500, the server puts the full traceback in the JSON
response (the local CLI prints it). You can also run the **sanity-check cell** below to
test inference directly. The server cell is **re-runnable** — fix a cell and re-run it
without restarting the runtime or the tunnel.

Response schema matches Docflow's `PageLayout` contract:
`{page_index, image_width, image_height, elements: [{category, bbox, text}]}`.

In [ ]:
# 1. Dependencies.
# dots.mocr's custom modeling code targets transformers==4.57.6. Other versions change
# generate() internals and break it (the 'mm_token_type_ids' and 'cache_position'
# TypeErrors). Pinning fixes both.
#
# IMPORTANT: Colab pre-imports a different transformers. After this cell finishes you MUST
# restart the runtime so the pinned version is the one that gets imported:
#   Runtime -> Restart session, then Run all.
!pip install -q "transformers==4.57.6" qwen_vl_utils accelerate huggingface_hub flask flask-cors pyngrok pillow


In [ ]:
# 2. Download dots.mocr (3B) weights to a dot-free folder.
# The HF repo id 'rednote-hilab/dots.mocr' contains a '.', which breaks transformers'
# trust_remote_code dynamic import. Downloading to ./DotsMOCR sidesteps that.
from huggingface_hub import snapshot_download

MODEL_DIR = snapshot_download(repo_id="rednote-hilab/dots.mocr", local_dir="./DotsMOCR")
print("weights at:", MODEL_DIR)


In [ ]:
# 2b. Make dots.mocr run WITHOUT flash-attn.
# flash-attn v2 needs Ampere+ (sm_80); Colab's free T4 is Turing (sm_75), so it can
# NEVER be installed here. The vision tower force-imports it at module top, but it also
# ships working SDPA/eager attention. Two steps switch it over:
#   (1) Stub the `flash_attn` module so the top-level import doesn't crash.
#   (2) Rewrite every flash_attention_2 -> sdpa in config.json (covers vision_config).
import importlib.abc
import importlib.machinery
import json
import os
import sys
import types


class _FlashStub(types.ModuleType):
    def __getattr__(self, name):
        def _dummy(*args, **kwargs):
            raise RuntimeError("flash_attn is stubbed (unsupported on T4); using SDPA.")
        return _dummy


class _FlashFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path, target=None):
        if fullname == "flash_attn" or fullname.startswith("flash_attn."):
            return importlib.machinery.ModuleSpec(fullname, self)
        return None

    def create_module(self, spec):
        return _FlashStub(spec.name)

    def exec_module(self, module):
        pass


if not any(isinstance(f, _FlashFinder) for f in sys.meta_path):
    sys.meta_path.insert(0, _FlashFinder())


def _to_sdpa(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == "attn_implementation" and v == "flash_attention_2":
                obj[k] = "sdpa"
            else:
                _to_sdpa(v)
    elif isinstance(obj, list):
        for x in obj:
            _to_sdpa(x)


cfg_path = os.path.join(MODEL_DIR, "config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
_to_sdpa(cfg)
with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)
print("flash_attn stubbed + config patched to sdpa")


In [ ]:
# 3. Load the model and processor. If this cell errors, STOP and fix it here —
# a failed load is the most common cause of a later 500 from /parse.
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoProcessor

# Guard: dots.mocr needs exactly 4.57.6. If this fails, you skipped the runtime
# restart after cell 1 — restart now (Runtime -> Restart session) and Run all.
assert transformers.__version__ == "4.57.6", (
    f"transformers is {transformers.__version__}, expected 4.57.6 — "
    "restart the runtime after running cell 1."
)


def _load(attn):
    return AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        attn_implementation=attn,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )


# Prefer SDPA (fast, T4-compatible); fall back to eager if the LLM rejects sdpa.
try:
    model = _load("sdpa")
    used = "sdpa"
except Exception as exc:
    print("sdpa load failed, falling back to eager:", repr(exc))
    model = _load("eager")
    used = "eager"

processor = AutoProcessor.from_pretrained(MODEL_DIR, trust_remote_code=True)
print("transformers", transformers.__version__, "| attn:", used, "| model on:", model.device)


In [ ]:
# 4. Inference: one PIL image -> list of layout-element dicts.
import json
import math
import re

import torch
from qwen_vl_utils import process_vision_info

# --- Memory controls (tune these if you OOM / want sharper OCR) -------------------
# On a T4 there is no flash-attn, so SDPA materializes an attention matrix that grows
# with the SQUARE of the vision-token count. Capping the image to a pixel budget keeps
# it in check. ~1.0M px (e.g. 868x1148) fits comfortably on a 16 GB T4. Lower MAX_PIXELS
# if you still OOM; raise it (or move to an Ampere GPU) for sharper text.
MAX_PIXELS = 1_000_000
MIN_PIXELS = 200_000
# Official dots.mocr default is 24000. We use less to bound T4 memory; raise toward
# 16000 if dense pages come back truncated (partial JSON), lower if you OOM.
MAX_NEW_TOKENS = 8192
FACTOR = 28  # vision patch(14) x 2x2 merge -> dims must be multiples of 28

# Keep the processor from re-resizing what we hand it (so our scale-back stays exact).
if hasattr(processor, "image_processor"):
    processor.image_processor.max_pixels = MAX_PIXELS
    processor.image_processor.min_pixels = MIN_PIXELS

# Exact official dots.mocr layout prompt (dots_mocr.utils.dict_promptmode_to_prompt).
PROMPT = """Please output the layout information from the PDF image, including each layout element's bbox, its category, and the corresponding text content within the bbox.

1. Bbox format: [x1, y1, x2, y2]

2. Layout Categories: The possible categories are ['Caption', 'Footnote', 'Formula', 'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', 'Title'].

3. Text Extraction & Formatting Rules:
    - Picture: For the 'Picture' category, the text field should be omitted.
    - Formula: Format its text as LaTeX.
    - Table: Format its text as HTML.
    - All Others (Text, Title, etc.): Format their text as Markdown.

4. Constraints:
    - The output text must be the original text from the image, with no translation.
    - All layout elements must be sorted according to human reading order.

5. Final Output: The entire output must be a single JSON object.
"""


def _smart_resize(w, h):
    """Round to multiples of FACTOR, keeping aspect, within [MIN_PIXELS, MAX_PIXELS].
    Mirrors the Qwen/dots vision pre-resize so the processor won't resize again."""
    w_bar = max(FACTOR, round(w / FACTOR) * FACTOR)
    h_bar = max(FACTOR, round(h / FACTOR) * FACTOR)
    if w_bar * h_bar > MAX_PIXELS:
        beta = math.sqrt((w * h) / MAX_PIXELS)
        w_bar = max(FACTOR, math.floor(w / beta / FACTOR) * FACTOR)
        h_bar = max(FACTOR, math.floor(h / beta / FACTOR) * FACTOR)
    elif w_bar * h_bar < MIN_PIXELS:
        beta = math.sqrt(MIN_PIXELS / (w * h))
        w_bar = math.ceil(w * beta / FACTOR) * FACTOR
        h_bar = math.ceil(h * beta / FACTOR) * FACTOR
    return w_bar, h_bar


def _to_elements(text):
    """Parse the model's decoded text into a list of element dicts.
    Tolerates code fences and either a JSON array or a {\"elements\": [...]} object."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n?|\n?```$", "", text).strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"(\[.*\]|\{.*\})", text, re.DOTALL)
        if not m:
            raise ValueError(f"No JSON found in model output: {text[:500]!r}")
        data = json.loads(m.group(0))
    if isinstance(data, dict):
        for key in ("elements", "layout", "results"):
            if isinstance(data.get(key), list):
                return data[key]
        return [data]
    return data


def parse_image(image):
    w0, h0 = image.size
    w1, h1 = _smart_resize(w0, h0)
    small = image.resize((w1, h1)) if (w1, h1) != (w0, h0) else image

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": small},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    # Newer transformers processors add 'mm_token_type_ids', which dots.mocr's
    # generate() rejects ("model_kwargs are not used by the model"). Drop it.
    inputs.pop("mm_token_type_ids", None)

    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    decoded = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    elements = _to_elements(decoded)

    # Scale bboxes from the resized space back to the ORIGINAL image pixels, so the
    # local engine's figure cropping (which assumes the original render dims) lines up.
    sx, sy = w0 / w1, h0 / h1
    for e in elements:
        if isinstance(e, dict) and e.get("bbox"):
            x1, y1, x2, y2 = e["bbox"]
            e["bbox"] = [x1 * sx, y1 * sy, x2 * sx, y2 * sy]

    torch.cuda.empty_cache()
    return elements


In [ ]:
# 4b. (Optional) Sanity-check inference directly — surfaces any error with a full
# traceback, without going through HTTP. Replace the blank image with a real page
# image for a meaningful result.
import traceback
from PIL import Image

try:
    out = parse_image(Image.new("RGB", (1000, 1400), "white"))
    print("OK -", type(out), "| first elements:", out[:2] if isinstance(out, list) else out)
except Exception:
    traceback.print_exc()

In [ ]:
# 5. Flask server exposing /parse and /health. Re-runnable: it shuts down any
# server started by a previous run of this cell before starting a new one.
import io
import threading
import traceback
from flask import Flask, jsonify, request
from flask_cors import CORS
from PIL import Image
from werkzeug.serving import make_server

try:
    _server.shutdown()  # noqa: F821 -- defined on a previous run
except NameError:
    pass

app = Flask(__name__)
CORS(app)


@app.get("/health")
def health():
    return jsonify(status="ok")


@app.post("/parse")
def parse():
    try:
        page_index = int(request.form.get("page_index", 0))
        image = Image.open(io.BytesIO(request.files["image"].read())).convert("RGB")
        elements = parse_image(image)
        norm = [
            {"category": e.get("category"), "bbox": e.get("bbox"), "text": e.get("text")}
            for e in elements
            if isinstance(e, dict)
        ]
        return jsonify(
            page_index=page_index,
            image_width=image.width,
            image_height=image.height,
            elements=norm,
        )
    except Exception as exc:
        tb = traceback.format_exc()
        print("ERROR in /parse:\n", tb)
        return jsonify(error=str(exc), traceback=tb), 500


_server = make_server("0.0.0.0", 5000, app, threaded=True)
threading.Thread(target=_server.serve_forever, daemon=True).start()
print("Flask serving on :5000 (re-runnable)")

## 6. Expose a public URL

Uses **pyngrok**. Get a free authtoken at <https://dashboard.ngrok.com> and paste it below.
You only need to run this **once** per session — if you re-run the server cell above, the
tunnel keeps pointing at port 5000.

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("PASTE_YOUR_NGROK_AUTHTOKEN_HERE")
public_url = ngrok.connect(5000).public_url
print("DOCFLOW_DOTS_URL =", public_url)
print("\nOn your machine (PowerShell):")
print(f'  $env:DOCFLOW_DOTS_URL="{public_url}"')
print("\nOn or in Command Prompt:")
print(f'  set DOCFLOW_DOTS_URL={public_url}')
print("  uv run docflow convert file.pdf --model dots -o out/")